<img src="VSM_logo.gif" width="500" align="center">

# VSM - Volcanic and Seismic source Modelling

**VSM - Volcanic and Seismic source Modelling** is a Python tool to perform inversions of geodetic data.

**VSM2** is the first major updated version of the original VSM software.

**Code** https://github.com/EliTras/VSM2

**License** E. Trasatti & F. Silverii - INGV (elisa.trasatti@ingv.it, francesca.silverii@ingv.it), covered by GNU-GPL License https://github.com/EliTras/VSM/blob/main/license.lic

This Notebook contains several utilities to interface with VSM. It can run VSM from an existing input file. It also contains code to compute volume variations, dislocation vertices and additional plots.
Make sure to run the first cell at the beginning. \
Use VSM_writeinput.ipynb for step-by-step instructions to create an input file.

----
Run the next cell in any case!

In [1]:
import os
import sys
sys.path.append('/Users/francescasilverii/VSM2')
import VSM
import VSM_utilities as VSMU

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import cartopy.crs as ccrs

from matplotlib import rcParams

import math

rcParams['pdf.fonttype'] = 42
rcParams['ps.fonttype'] = 42
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams["font.sans-serif"] = ['Calibri'] + plt.rcParams['font.sans-serif']
#plt.rcParams['font.family'] = 'serif'
#plt.rcParams['font.serif'] = ['Times New Roman'] + plt.rcParams['font.serif']

plt.rcParams.update({'font.size':14})

## Launch VSM

No need to run the following cells if you want to do post-processing. Just go to **Utilities**

#### Insert input file

In [ ]:
folder_work = '../USECASE/mogi1NA'

filename_in = 'VSM_input_mogi.txt'

#### Read all inputs to populate all variables of VSM

In [ ]:
VSM.read_VSM_settings(os.path.join(folder_work, filename_in))

#### Launch the execution of VSM

In [ ]:
VSM.iVSM()

----------
## Utilities

Here are some utilities to compute the volume variation of the sources from their shape and overpressure vs shear modulus ratio ($\Delta P/ \mu$). Also, it computes the vertices of the fault/dike.

In particular
- Volume variation of the finite volume sphere
- Volume variation of the finite volume spheroid
- Volume variation of the sill in point-source approximation
- Rectangular dislocation vertices and trace

Hint: just run the box you need!

#### Volume variation of the spheroid
Define semi-major axis, $\Delta P/ \mu$ and ratio between axes (ratio < 1)

In [2]:
a = 1300
dP_mu = 7.5e-3
ratio = 0.3

dVol = VSMU.volume_var_yang(dP_mu,a,ratio)
print('Volume variation of the finite volume spheroid',dVol/1e6, '1e6 m3')

Volume variation of the finite volume spheroid 5.544095004894214 10e6 m3


#### Volume variation of the sphere
Define radius, $\Delta P/ \mu$ and depth

In [3]:
radius = 800
dP_mu = 3.1e-3
depth = 2550
    
dVol = VSMU.volume_var_mctigue(dP_mu,radius,depth)
print('Volume variation of the finite volume sphere',dVol/1e6, '1e6 m3')

Volume variation of the finite volume sphere 5.034639599298008 10e6 m3


#### Volume variation of the penny-shaped crack
Define radius, $\Delta P/ \mu$ and Poisson Coefficient

In [4]:
# holds for depth/radius > 2 or 2.5all
radius = 820
dP_mu = 5.94e-3
Poisson = 0.25

dVol = VSMU.volume_var_penny(dP_mu,radius,Poisson)
print('Volume variation of the penny-shaped crack',dVol/1e6,' 1e6 m3')

Volume variation of the penny-shaped crack 6.55025184  10e6 m3


#### Vertices of the dislocation from the top left corner
Define Top Left Corner (TLC) coordinates, length, width, strike and dip of the plane. Obtain Bottom Left Corner (BLT), Bottom Right Corner (BRC), Top Right Corner (TRC), and the Trace Left & Right coordinates

In [5]:
TLC = [7.48873e+05,9.82e+06,2.21526e+03]
length = 9000.
width  = 4000.
strike = 180.
dip    = 83.
    
vert,trace = VSMU.fault_vertices(TLC,length,width,strike,dip)
print('Fault vertices ---------------------')
print('Fault TLC',vert[0])
print('Fault BLC',vert[1])
print('Fault BRC',vert[2])
print('Fault TRC',vert[3])
print('Trace ------------------------------')
print('Trace Left', trace[0])
print('Trace Right',trace[1])

Fault vertices ---------------------
Fault TLC [7.48873e+05 9.82000e+06 2.21526e+03]
Fault BLC [7.48385523e+05 9.82000000e+06 6.18544461e+03]
Fault BRC [7.48385523e+05 9.81100000e+06 6.18544461e+03]
Fault TRC [7.48873e+05 9.81100e+06 2.21526e+03]
Trace ------------------------------
Trace Left [ 749144.99972639 9820000.               0.        ]
Trace Right [ 749144.99972639 9811000.               0.        ]


### TPE utilities 

Estimation of some TPE parameters assuming values of others, and estimation of TPE volume variation

#### TPE - Estimate potency assuming others

In [9]:
H = 1e9    # poro-elastic parameter (Pa)
alpha = 3e-5 # thermal expansion coefficient (1/K)
dp = 10e6   # pore pressure change of the inclusion (Pa)
dT = 50    # Temperature change of the inclusion (K)

e0 = VSMU.TPE_potency(H, alpha, dp, dT)

print('Assuming H', H/1e9, 'GPa, alpha', alpha, '1/K, dp', dp/1e6, 'MPa, dT', dT,'K')
print('Potency: ', e0)


Assuming H 1.0 GPa, alpha 3e-05 1/K, dp 10.0 MPa, dT 50 K
Potency:  0.0038333333333333336


#### TPE - Estimate H assuming others

In [10]:
# Option 1 - estimate K from ni and mu
ni = 0.25       # Poisson coefficient
mu = 1e9        # shear modulus (Pa)
Ks = 30e9       # Unjacketed bulk modulus Ks (Pa)
K = (2*mu*(1+ni))/(3*(1-2*ni)) # Drained isothermal bulk modulus of the medium (Pa)

H = VSMU.TPE_H(K, Ks)
print('Assuming ni:', ni, ', mu:', mu/1e9, 'GPa, Ks', Ks/1e9,'GPa')
print('Drained isothermal bulk modulus K', K/1e9, 'GPa')
print('Poroelastic parameter H', H/1e9, 'GPa')


# Option 2 - set K 
K = 0.1e9       # Drained isothermal bulk modulus of the medium (Pa)
Ks = 30e9       # Unjacketed bulk modulus Ks

H = VSMU.TPE_H(K, Ks)
print('Assuming K:', K/1e9, 'GPa, Ks:', Ks/1e9,'GPa')
print('Poroelastic parameter H', H/1e9, 'GPa')


K calculated 1.6666666666666667
Assuming nu: 0.25 , mu 1.0 GPa, Ks 30.0 GPa
Drained isothermal bulk modulus K 0.1 GPa
Poroelastic parameter H 0.1003344481605351 GPa


#### TPE - Estimate overpressure with/without delta T

In [11]:
e0 = 1.61e-3    # potency
H = 1e9        # poro-elastic parameter (Pa)
alpha = 3e-5    # thermal expansion  coefficient (1/K)
dT = 50         # Temperature change of the inclusion (K)
    
dp1, dp2 = VSMU.TPE_dp(e0, alpha, H, dT)
    
print('\nAssuming e0', e0,', dT', dT, 'K, alpha', alpha,'1/K, H', H/1e9, 'GPa')
print('dP only   ', dp1/1e6,'MPa')
print('dP with dT', dp2/1e6,'MPa')



Assuming e0 0.00161 , dT 50 K, alpha 3e-05 1/K, H 1.0 GPa
dP only    4.83 MPa
dP with dT 3.3300000000000005 MPa


#### TPE - Volume variation of the cylinder

In [13]:
ni = 0.25       # Poisson coefficient
r = 1897        # cylinder radius (m)
h = 144         # cylinder height (m)
e0 = 1.61e-3    # potency

volume, dVolume = VSMU.TPE_volume_var_cylinder(e0,r,h,ni)

print('\nAssuming e0',e0,', radius',r,'m, height',h,'m')
print('Volume of the cylinder', volume/1e6,'1e6 m3')
print('Volume variation of the cylinder', dVolume/1e6,'1e6 m3')



Assuming e0 0.00161 , radius 1897 m, height 144 m
Volume of the cylinder 1627.972358046064 1e6 m3
Volume variation of the cylinder 4.368392494090272 1e6 m3


#### TPE - Volume variation of the sphere

In [15]:
ni = 0.25       # Poisson coefficient
r = 319         # sphere radius (m)
e0 = 1.61e-3    # potency

volume, dVolume = VSMU.TPE_volume_var_sphere(e0,r,ni)

print('\nAssuming e0',e0,', radius',r,'m')
print('Volume of the sphere', volume/1e6,'1e6 m3')
print('Volume variation of the sphere', dVolume/1e6,'1e6 m3')



Assuming e0 0.00161 , radius 319 m
Volume of the sphere 135.97549812933644 1e6 m3
Volume variation of the sphere 0.3648675866470528 1e6 m3


----
## Plots

Here are two code blocks to plot
- 1D and 2D statistics of the parameters searched using the corner library. May be used to change the number of burn-in models
- Quick plot of the comparisons among oberved data, synthetic data and residuals.

#### Plot 1D and 2D statistics
It creates the same plot as VSM. May be used to change the number of burn-in models.

Should set
- working folder
- file with the ensample of models generated
- file of mean or best model (based on the inversion chosen in VSM)
- burn-in models

In [ ]:
# PLOT 1D - 2D statistics via corner
folder = '../USECASE/mogi1NA'
f_models ='VSM_models.csv'
# best (for NA) or mean (for BI)
f_truths = 'VSM_best.csv'
f_plot = 'VSM_plot1D2D_new.png'  #can use pdf or png extension
nskip = 1000

df = pd.read_csv(os.path.join(folder,f_models))
tf = pd.read_csv(os.path.join(folder,f_truths))
nunk = np.size(tf)
    
if(f_truths[4:8] == 'best'):
    dd = df.iloc[:,2:] # NA
    tt = tf.values.reshape(nunk) #NA
else:
    dd = df #BI
    tp = tf.values.reshape(nunk)
    tt = tp[::2] #BI
    
VSM.plot_1D_2D(dd, tt, dd.columns, os.path.join(folder,f_plot), nskip)

#### Plot Data, model and residuals

The following code creates 3 plots automatically scaled to data extension. The figure is composed of three panels: observed data, modelled data (from optimal model) and residuals, e.g., data minus modelled data.
Should set
- working folder
- file with synthetic data. It works for SAR and levelling data as it takes the third column of the synth file as modelled data. Should be rearranged for other geodetic data.
- UTM zone

In [ ]:
folder_inout = '../USECASE/mogi1NA'
synth_file = 'VSM_synth_sar2.csv'
out_file = 'VSM_res_sar2.png'

# UTM zone
zone = 33
southern_hemisphere = False

In [ ]:
# My data in UTM coordinates
db_sar = pd.read_csv(os.path.join(folder_inout,synth_file))
d_sar = db_sar.values

# Split into east and north coordinates
east, north = d_sar[:,0],d_sar[:,1]
data = d_sar[:,3]
synth = d_sar[:,2]
res = data - synth

dmax = max(max(data),max(synth))
dmin = min(min(data),min(synth))
resmax = max(res)
resmin = min(res)
resext = max(resmax, -resmin)

# Define the projection
#crs=ccrs.PlateCarree()
mycrs=ccrs.UTM(zone=zone, southern_hemisphere=southern_hemisphere)

fig=plt.figure(figsize=(15,6))
palette='viridis' #'RdYlBu_r'

## PANEL DATA ##########
ax = plt.subplot(131, projection=mycrs)
ax.coastlines(resolution='10m')
img = ax.scatter(east, north,5, data,cmap=palette, vmin=dmin, vmax = dmax)
#palette
cbar = plt.colorbar(img,orientation='horizontal')
cbar.set_label('LOS (m)')
# Get the extent of the axis
extent = ax.get_extent()
# Attempt to set the axis extent
ax.set_extent(extent, crs=mycrs)
plt.title('Data',fontsize = 16, pad=10)

## PANEL MODEL ##########
ax = plt.subplot(132, projection=mycrs)
ax.coastlines(resolution='10m')
img = ax.scatter(east, north,5, synth,cmap=palette, vmin=dmin, vmax = dmax)
#palette
cbar = plt.colorbar(img,orientation='horizontal')
cbar.set_label('LOS (m)')
# Get the extent of the axis
extent = ax.get_extent()
# Attempt to set the axis extent
ax.set_extent(extent, crs=mycrs)
plt.title('Model',fontsize = 16, pad=10)

## PANEL RESIDUALS ##########
ax = plt.subplot(133, projection=mycrs)
ax.coastlines(resolution='10m')
img = ax.scatter(east, north,5, res,cmap="bwr",vmin=resext, vmax = -resext)
#palette
cbar = plt.colorbar(img,orientation='horizontal')
cbar.set_label('LOS (m)')
# Get the extent of the axis
extent = ax.get_extent()
# Attempt to set the axis extent
ax.set_extent(extent, crs=mycrs)
# Title for plot
plt.title('Residual',fontsize = 16, pad=10)

plt.savefig(os.path.join(folder_inout,out_file))
plt.show()